# PCA Generalizability Diagnostics

This notebook is a diagnostic companion to `generalizability_figure_pca.ipynb`.

The original PCA score can saturate because the copy threshold is calibrated from real-real nearest-neighbor cosine similarities. In these CAMELS slices, real-real nearest neighbors can be extremely close in a low-dimensional PCA space, so the 99% threshold can sit near 1.0. Then generated samples rarely cross the threshold, giving `copy_fraction = 0` and `PCA_GL = 1` for every dataset size.

This notebook keeps the original threshold idea, but adds the checks needed to understand whether the score is informative:

- generated-to-training nearest-neighbor cosine distributions
- real-validation-to-training nearest-neighbor cosine distributions
- real-training leave-one-out nearest-neighbor thresholds at q=0.90, 0.95, and 0.99
- generated versus nearest-training image pairs

Interpret this as a memorization/novelty diagnostic only. Use one-point statistics, P(k), and image inspection for sample quality.


## Configuration

Useful environment variables before launching the notebook on Great Lakes:

```bash
export GENERALIZABILITY_ARCHES=u64
export PCA_N_COMPONENTS=32
export PCA_COPY_QUANTILES=0.90,0.95,0.99
export PCA_MAX_FIT_REAL=1024
export PCA_MAX_REAL_COMPARE=4096
export PCA_MAX_GENERATED=128
```

`PCA_FIT_MODE="largest"` fits one PCA basis per architecture using the largest available real training set, then reuses that basis across dataset sizes. This is usually the cleaner comparison.


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/home/jiamingp/Diffusion_model'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])

for candidate in (PROJECT_DIR, PROJECT_DIR / 'cosmo_diffusion'):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from simdiff_eval.io import as_nchw, load_npy, load_real_from_config

ARCHES = [x.strip() for x in os.environ.get('GENERALIZABILITY_ARCHES', 'u64').split(',') if x.strip()]
ARCH_LABELS = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256'}
ARCH_COLORS = {'u64': 'tab:red', 'u128': 'tab:blue', 'u256': 'limegreen'}
ARCH_MARKERS = {'u64': 'o', 'u128': 's', 'u256': '^'}

SEED = int(os.environ.get('GENERALIZABILITY_SEED', 123))
PCA_N_COMPONENTS = int(os.environ.get('PCA_N_COMPONENTS', 32))
PCA_MAX_FIT_REAL = int(os.environ.get('PCA_MAX_FIT_REAL', 1024))
PCA_MAX_REAL_COMPARE = int(os.environ.get('PCA_MAX_REAL_COMPARE', 4096))
PCA_MAX_GENERATED = None if os.environ.get('PCA_MAX_GENERATED', '').strip() == '' else int(os.environ['PCA_MAX_GENERATED'])
PCA_COPY_QUANTILES = tuple(float(x) for x in os.environ.get('PCA_COPY_QUANTILES', '0.90,0.95,0.99').split(','))
PCA_FIT_MODE = os.environ.get('PCA_FIT_MODE', 'largest')  # largest or per_run
PCA_REF_FRACTION = float(os.environ.get('PCA_REF_FRACTION', 0.75))
SIMILARITY_BATCH_SIZE = int(os.environ.get('PCA_SIMILARITY_BATCH_SIZE', 1024))
N_PAIR_EXAMPLES = int(os.environ.get('PCA_N_PAIR_EXAMPLES', 6))

MANIFEST_CANDIDATES = [
    PROJECT_DIR / 'local' / 'fig1_lh' / 'manifest.json',
    PROJECT_DIR / 'configs' / 'templates' / 'reproducibility_manifest_template.json',
]
MANIFEST_PATH = next((p for p in MANIFEST_CANDIDATES if p.exists()), MANIFEST_CANDIDATES[0])
CONFIG_DIR = PROJECT_DIR / 'local' / 'fig1_lh' / 'configs'
SAMPLE_ROOTS = [
    PROJECT_DIR / 'results' / 'fig1_lh' / 'samples',
    PROJECT_DIR / 'results' / 'tables' / 'samples',
    PROJECT_DIR / 'results' / 'samples',
]
OUTPUT_DIR = PROJECT_DIR / 'results' / 'figures'
TABLE_DIR = PROJECT_DIR / 'results' / 'tables'

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

print('project:', PROJECT_DIR)
print('manifest:', MANIFEST_PATH)
print('arches:', ARCHES)
print('PCA components:', PCA_N_COMPONENTS)
print('PCA fit mode:', PCA_FIT_MODE)
print('PCA copy quantiles:', PCA_COPY_QUANTILES)
print('PCA fit slice cap:', PCA_MAX_FIT_REAL)
print('real comparison slice cap:', PCA_MAX_REAL_COMPARE)
print('generated cap:', PCA_MAX_GENERATED)
print('sample roots:')
for root in SAMPLE_ROOTS:
    print(' ', root, 'exists=', root.exists())


## Manifest And Sample Discovery

This uses the same `local/fig1_lh/manifest.json` rows as the original generalizability figure. Missing samples are skipped.


In [ ]:
def dataset_size(row: dict[str, Any]) -> float:
    for key in ('dataset_size', 'actual_2d', 'target_2d'):
        value = row.get(key)
        if value is not None:
            return float(value)
    raise ValueError(f'No dataset-size field in row: {row}')


def config_path_for(row: dict[str, Any]) -> Path:
    if row.get('config'):
        path = Path(row['config'])
        if not path.is_absolute():
            path = PROJECT_DIR / path
        return path
    return CONFIG_DIR / f"{row['run_name']}.yaml"


def sample_path_for(row: dict[str, Any]) -> Path | None:
    if row.get('sample_path'):
        raw = str(row['sample_path']).format(seed=SEED, run_name=row['run_name'])
        path = Path(raw)
        if not path.is_absolute():
            path = PROJECT_DIR / path
        if path.exists():
            return path
    for root in SAMPLE_ROOTS:
        for suffix in ('.npy', '.npz'):
            path = root / f"{row['run_name']}_seed{SEED}{suffix}"
            if path.exists():
                return path
    return None


def raw_sim_cap_for(row: dict[str, Any], slice_cap: int | None) -> int | None:
    if slice_cap is None:
        return None
    slices_per_sim = row.get('slices_per_sim')
    if slices_per_sim is None:
        zthin = int(row.get('zthin', 4) or 4)
        slices_per_sim = max(1, 128 // zthin)
    cap = max(1, int(math.ceil(int(slice_cap) / int(slices_per_sim))))
    total_raw = row.get('n_samples_simulations')
    if total_raw is not None:
        cap = min(cap, int(total_raw))
    return cap


def load_sample_array(path: Path) -> np.ndarray:
    if path.suffix == '.npz':
        z = np.load(path, mmap_mode='r')
        try:
            if 'samples' in z:
                return np.asarray(z['samples'])
            if 'arr_0' in z:
                return np.asarray(z['arr_0'])
            return np.asarray(z[z.files[0]])
        finally:
            z.close()
    return load_npy(path)


manifest = json.loads(MANIFEST_PATH.read_text())
rows = sorted([row for row in manifest if row.get('arch') in ARCHES], key=lambda row: (row.get('arch', ''), dataset_size(row)))
if not rows:
    raise RuntimeError(f'No manifest rows found for ARCHES={ARCHES!r}')

status_rows = []
for row in rows:
    config_path = config_path_for(row)
    sample_path = sample_path_for(row)
    status_rows.append({
        'arch': row.get('arch'),
        'run_name': row['run_name'],
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'real_compare_raw_sim_cap': raw_sim_cap_for(row, PCA_MAX_REAL_COMPARE),
        'pca_fit_raw_sim_cap': raw_sim_cap_for(row, PCA_MAX_FIT_REAL),
        'config_exists': config_path.exists(),
        'sample_exists': sample_path is not None,
        'config_path': str(config_path),
        'sample_path': str(sample_path) if sample_path is not None else None,
    })

status_df = pd.DataFrame(status_rows).sort_values(['arch', 'dataset_size']).reset_index(drop=True)
display(status_df)
print('available sample rows:', int(status_df['sample_exists'].sum()), '/', len(status_df))


## PCA Encoder And Similarity Helpers

Embeddings are flattened normalized fields projected onto PCA components and then L2-normalized. Cosine similarity is computed in this PCA space.

For each generated sample `j`, the training-neighbor statistic is explicitly

` s_j = max_i cos(z_gen[j], z_train[i]) `

where `z_train` is the PCA embedding of the reference/training pool. The same max-over-reference statistic is also computed for held-out real validation slices.

For each row we split real data into:

- `real_ref`: the reference/training pool used for nearest-neighbor lookup
- `real_val`: held-out real slices from the same loaded real pool

The validation-to-reference distribution is a sanity baseline: if generated-to-reference max similarities are lower than validation-to-reference max similarities, the generated samples are not unusually close to training in this PCA space.


In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return arr[idx].copy()


def flatten_images(images: np.ndarray) -> np.ndarray:
    arr = as_nchw(images).astype(np.float32, copy=False)
    return arr.reshape(len(arr), -1)


class PCAEncoder:
    def __init__(self, mean: np.ndarray, scale: np.ndarray, components: np.ndarray, explained_variance_ratio: np.ndarray):
        self.mean = mean.astype(np.float32)
        self.scale = scale.astype(np.float32)
        self.components = components.astype(np.float32)
        self.explained_variance_ratio = explained_variance_ratio.astype(np.float32)

    def transform(self, images: np.ndarray) -> np.ndarray:
        x = flatten_images(images)
        x = (x - self.mean) / self.scale
        return x @ self.components.T


def fit_pca_encoder(real: np.ndarray, n_components: int = 32, max_fit: int = 1024) -> PCAEncoder:
    x = flatten_images(evenly_limit(real, max_fit))
    mean = x.mean(axis=0, keepdims=True)
    scale = x.std(axis=0, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)
    x = ((x - mean) / scale).astype(np.float32, copy=False)
    n_components = int(min(n_components, x.shape[0] - 1, x.shape[1]))
    if n_components < 2:
        raise ValueError('Need at least 3 real samples to fit PCA.')

    try:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=n_components, svd_solver='randomized', random_state=0)
        pca.fit(x)
        components = pca.components_
        evr = pca.explained_variance_ratio_
    except Exception as exc:
        print('sklearn PCA unavailable or failed; using numpy SVD:', repr(exc))
        _, s, vt = np.linalg.svd(x, full_matrices=False)
        components = vt[:n_components]
        var = (s ** 2) / max(len(x) - 1, 1)
        evr = var[:n_components] / np.clip(var.sum(), 1e-30, None)
    return PCAEncoder(mean.squeeze(0), scale.squeeze(0), components, evr)


def l2_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norm, 1e-12, None)


def nearest_self_similarity(z: np.ndarray, batch_size: int = 1024) -> tuple[np.ndarray, np.ndarray]:
    z = np.asarray(z, dtype=np.float32)
    sims = []
    idxs = []
    n = len(z)
    for start in range(0, n, batch_size):
        stop = min(start + batch_size, n)
        sim = z[start:stop] @ z.T
        rows = np.arange(stop - start)
        cols = np.arange(start, stop)
        sim[rows, cols] = -np.inf
        idx = np.argmax(sim, axis=1)
        sims.append(sim[np.arange(stop - start), idx])
        idxs.append(idx)
    return np.concatenate(sims), np.concatenate(idxs)


def nearest_cross_similarity(query_z: np.ndarray, ref_z: np.ndarray, batch_size: int = 1024) -> tuple[np.ndarray, np.ndarray]:
    sims = []
    idxs = []
    for start in range(0, len(query_z), batch_size):
        stop = min(start + batch_size, len(query_z))
        sim = query_z[start:stop] @ ref_z.T
        idx = np.argmax(sim, axis=1)
        sims.append(sim[np.arange(stop - start), idx])
        idxs.append(idx)
    return np.concatenate(sims), np.concatenate(idxs)


def deterministic_real_split(real: np.ndarray, ref_fraction: float = 0.75) -> tuple[np.ndarray, np.ndarray]:
    real = as_nchw(real)
    n = len(real)
    if n < 4:
        raise ValueError(f'Need at least 4 real slices for ref/validation split, got {n}.')
    n_ref = int(round(n * ref_fraction))
    n_ref = min(max(n_ref, 2), n - 2)
    idx = np.arange(n)
    # Evenly interleave the validation set so it is not just the final contiguous block.
    val_mask = (idx % max(2, int(round(1 / max(1e-6, 1 - ref_fraction))))) == 0
    if val_mask.sum() < 2 or (~val_mask).sum() < 2:
        val_mask = idx >= n_ref
    return real[~val_mask].copy(), real[val_mask].copy()


def quantile_metrics(values: np.ndarray, prefix: str) -> dict[str, float]:
    values = np.asarray(values)
    return {
        f'{prefix}_median': float(np.median(values)),
        f'{prefix}_q90': float(np.quantile(values, 0.90)),
        f'{prefix}_q95': float(np.quantile(values, 0.95)),
        f'{prefix}_q99': float(np.quantile(values, 0.99)),
    }


## Load Real Data And Fit PCA

For `PCA_FIT_MODE="largest"`, one PCA encoder is fitted per architecture from the largest completed row. This avoids changing the feature space at every dataset size.


In [ ]:
real_cache: dict[tuple[str, int | None], np.ndarray] = {}
encoder_cache: dict[str, dict[str, Any]] = {}


def load_real_for_row(row: dict[str, Any], *, slice_cap: int | None) -> np.ndarray:
    key = (row['run_name'], slice_cap)
    if key in real_cache:
        return real_cache[key]
    config_path = config_path_for(row)
    raw_cap = raw_sim_cap_for(row, slice_cap)
    real = load_real_from_config(config_path, max_raw_samples=raw_cap)
    real = evenly_limit(as_nchw(real), slice_cap)
    real_cache[key] = real
    return real


if PCA_FIT_MODE not in {'largest', 'per_run'}:
    raise ValueError('PCA_FIT_MODE must be "largest" or "per_run".')

if PCA_FIT_MODE == 'largest':
    for arch in ARCHES:
        arch_rows = [row for row in rows if row.get('arch') == arch and sample_path_for(row) is not None and config_path_for(row).exists()]
        if not arch_rows:
            print('No completed rows for PCA fit:', arch)
            continue
        fit_row = max(arch_rows, key=dataset_size)
        print(f"Fitting {arch} PCA on {fit_row['run_name']} with up to {PCA_MAX_FIT_REAL} real slices")
        fit_real = load_real_for_row(fit_row, slice_cap=PCA_MAX_FIT_REAL)
        encoder = fit_pca_encoder(fit_real, n_components=PCA_N_COMPONENTS, max_fit=PCA_MAX_FIT_REAL)
        encoder_cache[arch] = {
            'encoder': encoder,
            'fit_run_name': fit_row['run_name'],
            'fit_dataset_size': dataset_size(fit_row),
            'n_fit_real': len(fit_real),
        }
        print(
            f"  components={len(encoder.explained_variance_ratio)} "
            f"explained_var_sum={encoder.explained_variance_ratio.sum():.3f}"
        )


## Compute Diagnostic Table

The important columns are:

- `gen_nn_*`: generated-to-training nearest-neighbor cosine statistics, i.e. statistics of `s_j = max_i cos(z_gen[j], z_train[i])`
- `gen_max_train_sim_*`: explicit aliases for the same generated max-over-training statistic
- `val_nn_*`: real-validation-to-training nearest-neighbor cosine statistics, also max-over-training/reference
- `ref_nn_q*`: real-training leave-one-out nearest-neighbor thresholds
- `gen_copy_frac_q*`: fraction of generated samples above that real-real threshold
- `val_copy_frac_q*`: same fraction for held-out real validation samples

If `gen_copy_frac_q99` is always zero while `ref_nn_q99` is near one, the old PCA score is saturated and not very useful.


In [ ]:
records = []
similarity_cache: dict[str, dict[str, np.ndarray]] = {}

for row in rows:
    run_name = row['run_name']
    arch = row.get('arch')
    sample_path = sample_path_for(row)
    config_path = config_path_for(row)
    if sample_path is None or not config_path.exists():
        print('SKIP missing inputs:', run_name, 'sample=', sample_path, 'config exists=', config_path.exists())
        continue

    generated = as_nchw(load_sample_array(sample_path))
    generated = evenly_limit(generated, PCA_MAX_GENERATED)
    real_all = load_real_for_row(row, slice_cap=PCA_MAX_REAL_COMPARE)
    real_ref, real_val = deterministic_real_split(real_all, PCA_REF_FRACTION)

    if PCA_FIT_MODE == 'largest':
        if arch not in encoder_cache:
            print('SKIP no PCA encoder for arch:', arch)
            continue
        encoder_info = encoder_cache[arch]
        encoder = encoder_info['encoder']
    else:
        print(f'Fitting per-run PCA on {run_name}')
        fit_real = load_real_for_row(row, slice_cap=PCA_MAX_FIT_REAL)
        encoder = fit_pca_encoder(fit_real, n_components=PCA_N_COMPONENTS, max_fit=PCA_MAX_FIT_REAL)
        encoder_info = {'fit_run_name': run_name, 'fit_dataset_size': dataset_size(row), 'n_fit_real': len(fit_real)}

    print(
        f"{run_name}: arch={arch} gen={len(generated)} real_ref={len(real_ref)} "
        f"real_val={len(real_val)} dataset_size={dataset_size(row):.0f}"
    )

    ref_z = l2_normalize(encoder.transform(real_ref))
    val_z = l2_normalize(encoder.transform(real_val))
    gen_z = l2_normalize(encoder.transform(generated))

    ref_nn, ref_nn_idx = nearest_self_similarity(ref_z, SIMILARITY_BATCH_SIZE)
    val_nn, val_match_idx = nearest_cross_similarity(val_z, ref_z, SIMILARITY_BATCH_SIZE)
    gen_nn, gen_match_idx = nearest_cross_similarity(gen_z, ref_z, SIMILARITY_BATCH_SIZE)

    finite_ref = ref_nn[np.isfinite(ref_nn)]
    rec = {
        'run_name': run_name,
        'arch': arch,
        'arch_label': ARCH_LABELS.get(arch, arch),
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'n_generated': len(generated),
        'n_real_ref': len(real_ref),
        'n_real_val': len(real_val),
        'pca_components': len(encoder.explained_variance_ratio),
        'pca_explained_variance_sum': float(encoder.explained_variance_ratio.sum()),
        'pca_fit_mode': PCA_FIT_MODE,
        'pca_fit_run_name': encoder_info['fit_run_name'],
        'pca_fit_dataset_size': encoder_info['fit_dataset_size'],
        'n_pca_fit_real': encoder_info['n_fit_real'],
        'real_slice_cap': PCA_MAX_REAL_COMPARE,
        'generated_slice_cap': PCA_MAX_GENERATED,
        'sample_path': str(sample_path),
        **quantile_metrics(finite_ref, 'ref_nn'),
        **quantile_metrics(val_nn, 'val_nn'),
        **quantile_metrics(gen_nn, 'gen_nn'),
    }

    for stat in ('median', 'q90', 'q95', 'q99'):
        rec[f'gen_max_train_sim_{stat}'] = rec[f'gen_nn_{stat}']
        rec[f'val_max_train_sim_{stat}'] = rec[f'val_nn_{stat}']

    for q in PCA_COPY_QUANTILES:
        threshold = float(np.quantile(finite_ref, q))
        key = int(round(q * 100))
        rec[f'threshold_q{key}'] = threshold
        rec[f'gen_copy_frac_q{key}'] = float(np.mean(gen_nn >= threshold))
        rec[f'val_copy_frac_q{key}'] = float(np.mean(val_nn >= threshold))

    rec['gen_minus_val_nn_median'] = rec['gen_nn_median'] - rec['val_nn_median']
    rec['gen_q99_minus_ref_q99'] = rec['gen_nn_q99'] - rec['ref_nn_q99']
    records.append(rec)

    similarity_cache[run_name] = {
        'real_ref': real_ref,
        'real_val': real_val,
        'generated': generated,
        'ref_nn': ref_nn,
        'val_nn': val_nn,
        'gen_nn': gen_nn,
        'gen_match_idx': gen_match_idx,
    }

    qtext = ' '.join([f"q{int(round(q*100))}: gen={rec[f'gen_copy_frac_q{int(round(q*100))}']:.3f}, val={rec[f'val_copy_frac_q{int(round(q*100))}']:.3f}" for q in PCA_COPY_QUANTILES])
    print(
        f"  gen_median={rec['gen_nn_median']:.3f} val_median={rec['val_nn_median']:.3f} "
        f"ref_q99={rec['ref_nn_q99']:.3f} {qtext}"
    )

if not records:
    raise RuntimeError('No PCA diagnostic records computed. Check missing samples/configs above.')

df = pd.DataFrame(records).sort_values(['arch', 'dataset_size']).reset_index(drop=True)
display(df)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
out_csv = TABLE_DIR / 'figure2b_style_generalizability_pca_diagnostic_metrics.csv'
df.to_csv(out_csv, index=False)
print('saved', out_csv)


## Threshold Sweep: Generated Versus Real Validation

Each panel uses a different real-real nearest-neighbor threshold. If generated and validation both stay below the high threshold, then the binary copy score is probably too conservative.


In [ ]:
def xfmt(x: float, _pos: int) -> str:
    if x <= 0:
        return ''
    exponent = int(round(np.log2(x)))
    if np.isclose(x, 2**exponent):
        return rf'$2^{{{exponent}}}$'
    return f'{x:g}'

nq = len(PCA_COPY_QUANTILES)
fig, axes = plt.subplots(1, nq, figsize=(5.2 * nq, 4.3), sharey=True, squeeze=False)
plot_df = df.sort_values(['arch', 'dataset_size'])

for ax, q in zip(axes.ravel(), PCA_COPY_QUANTILES):
    key = int(round(q * 100))
    for arch in ARCHES:
        sub = plot_df[plot_df['arch'] == arch].sort_values('dataset_size')
        if sub.empty:
            continue
        color = ARCH_COLORS.get(arch)
        label = ARCH_LABELS.get(arch, arch)
        ax.plot(sub['dataset_size'], sub[f'gen_copy_frac_q{key}'], color=color, marker='o', lw=2.3, label=f'{label} generated')
        ax.plot(sub['dataset_size'], sub[f'val_copy_frac_q{key}'], color=color, marker='x', ls='--', lw=1.8, alpha=0.8, label=f'{label} real validation')
    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
    ax.set_title(f'above real-real q={q:g}')
    ax.set_xlabel('dataset size')
    ax.grid(alpha=0.25)
    ax.set_ylim(-0.05, 1.05)
axes[0, 0].set_ylabel('fraction above threshold')
axes[0, -1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
fig.suptitle('PCA nearest-neighbor copy fractions: generated vs real validation', y=1.03)
fig.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUTPUT_DIR / 'figure2b_style_generalizability_pca_threshold_sweep.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()


## Similarity Curves

The right way to debug the saturated score is to look at the continuous max-over-training similarities, not only the binary copy fraction.

For each generated sample this plots `s_j = max_i cos(z_gen[j], z_train[i])`. Higher cosine means the sample is closer to some training/reference slice in PCA space. If generated similarity is below real-validation similarity, the generated samples are not unusually training-like under this encoder.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.8), sharex=True)
plot_df = df.sort_values(['arch', 'dataset_size'])

for arch in ARCHES:
    sub = plot_df[plot_df['arch'] == arch].sort_values('dataset_size')
    if sub.empty:
        continue
    color = ARCH_COLORS.get(arch)
    label = ARCH_LABELS.get(arch, arch)
    axes[0].plot(sub['dataset_size'], sub['gen_nn_median'], color=color, marker='o', lw=2.5, label=f'{label} generated max_i median')
    axes[0].plot(sub['dataset_size'], sub['val_nn_median'], color=color, marker='x', ls='--', lw=2.0, label=f'{label} real-val max_i median')
    axes[1].plot(sub['dataset_size'], sub['gen_nn_q99'], color=color, marker='o', lw=2.5, label=f'{label} generated q99')
    axes[1].plot(sub['dataset_size'], sub['ref_nn_q99'], color=color, marker='s', ls=':', lw=2.0, label=f'{label} real-real q99 threshold')

for ax in axes:
    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
    ax.set_xlabel('dataset size')
    ax.set_ylabel('PCA nearest-neighbor cosine')
    ax.grid(alpha=0.25)
axes[0].set_title('median max-over-training similarity')
axes[1].set_title('generated max_i q99 versus real-real q99 threshold')
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
fig.tight_layout()
out = OUTPUT_DIR / 'figure2b_style_generalizability_pca_similarity_curves.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()


## Similarity Distributions For Small / Middle / Large Dataset Sizes

These histograms show why a single high quantile can saturate. The black distribution is real-reference leave-one-out similarity; the gray dashed lines are its thresholds. Blue is generated-to-reference using `s_j = max_i cos(z_gen[j], z_train[i])`. Orange is real-validation-to-reference using the same max-over-reference rule.


In [ ]:
def choose_representative_rows(frame: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for arch in ARCHES:
        sub = frame[frame['arch'] == arch].sort_values('dataset_size')
        if sub.empty:
            continue
        idxs = [0, len(sub) // 2, len(sub) - 1]
        pieces.append(sub.iloc[sorted(set(idxs))])
    return pd.concat(pieces, ignore_index=True) if pieces else frame.iloc[:0]

chosen = choose_representative_rows(df)
if chosen.empty:
    raise RuntimeError('No rows available for histogram diagnostics.')

fig, axes = plt.subplots(len(chosen), 1, figsize=(9.2, 3.2 * len(chosen)), squeeze=False)
for ax, (_, row) in zip(axes.ravel(), chosen.iterrows()):
    cache = similarity_cache[row['run_name']]
    values = np.concatenate([cache['ref_nn'], cache['val_nn'], cache['gen_nn']])
    bins = np.linspace(max(-1, np.nanmin(values) - 0.02), min(1, np.nanmax(values) + 0.02), 50)
    ax.hist(cache['ref_nn'], bins=bins, density=True, histtype='step', color='black', lw=2, label='real ref leave-one-out')
    ax.hist(cache['val_nn'], bins=bins, density=True, histtype='step', color='tab:orange', lw=2, label='real validation -> ref')
    ax.hist(cache['gen_nn'], bins=bins, density=True, histtype='step', color='tab:blue', lw=2, label='generated -> ref')
    for q in PCA_COPY_QUANTILES:
        key = int(round(q * 100))
        ax.axvline(row[f'threshold_q{key}'], color='gray', ls='--', lw=1.2, alpha=0.75)
    ax.set_title(f"{row['arch']} {row['dataset_tag']} N={row['dataset_size']:.0f}: PCA nearest-neighbor similarity")
    ax.set_xlabel('cosine similarity')
    ax.set_ylabel('density')
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.tight_layout()
out = OUTPUT_DIR / 'figure2b_style_generalizability_pca_similarity_histograms.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()


## Nearest Training Image Pairs

For the selected rows, this plots the generated samples with the highest PCA nearest-neighbor cosine and the training/reference slice they matched. This is the most direct way to see whether high similarity looks like copying or just similar morphology.


In [ ]:
def image_limits(*arrays: np.ndarray) -> tuple[float, float]:
    vals = np.concatenate([np.asarray(a).ravel() for a in arrays])
    lo, hi = np.nanpercentile(vals, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return -1.0, 1.0
    return float(lo), float(hi)

for _, row in chosen.iterrows():
    cache = similarity_cache[row['run_name']]
    gen = cache['generated']
    ref = cache['real_ref']
    sim = cache['gen_nn']
    match_idx = cache['gen_match_idx']
    order = np.argsort(sim)[::-1][:min(N_PAIR_EXAMPLES, len(sim))]
    vmin, vmax = image_limits(gen[order, 0], ref[match_idx[order], 0])
    fig, axes = plt.subplots(2, len(order), figsize=(2.3 * len(order), 4.8), squeeze=False)
    for col, gi in enumerate(order):
        ri = match_idx[gi]
        axes[0, col].imshow(gen[gi, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[0, col].set_title(f'gen #{gi}\ncos={sim[gi]:.3f}', fontsize=10)
        axes[1, col].imshow(ref[ri, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[1, col].set_title(f'nearest real #{ri}', fontsize=10)
        for ax in (axes[0, col], axes[1, col]):
            ax.set_xticks([])
            ax.set_yticks([])
    axes[0, 0].set_ylabel('generated')
    axes[1, 0].set_ylabel('nearest real')
    fig.suptitle(f"{row['arch']} {row['dataset_tag']} N={row['dataset_size']:.0f}: top PCA nearest-neighbor pairs", y=1.02)
    fig.tight_layout()
    safe_tag = str(row['dataset_tag']).replace('/', '_')
    out = OUTPUT_DIR / f"figure2b_style_generalizability_pca_pairs_{row['arch']}_{safe_tag}.png"
    fig.savefig(out, bbox_inches='tight')
    print('saved', out)
    plt.show()


## Quick Interpretation Checklist

Use this after running the notebook:

1. If `ref_nn_q99` is close to 1 and `gen_copy_frac_q99` is 0 everywhere, the old PCA GL curve is saturated.
2. If generated-to-reference similarities are below real-validation-to-reference similarities, generated samples are not unusually close to training in PCA space.
3. If generated similarities rise with dataset size but stay below the real-real threshold, the model may be learning the data manifold without literal nearest-neighbor copying.
4. If nearest-pair images look visually different even at high cosine, PCA cosine is probably measuring broad morphology rather than memorization.
5. If generated images look bad while PCA GL is high, the issue is sample quality, not memorization. Use one-point statistics and P(k) for that.
